# Week 8: On-Device Computer Vision with MediaPipe - Day 1

## 🗓️ 학습 주제: Theme 22 - MediaPipe 기반 인체 랜드마크 검출 및 분석

본 실습에서는 Google의 대표적인 경량 온디바이스 AI 프레임워크인 **MediaPipe**를 활용하여 얼굴, 손, 전신 관절을 추적하고, 이를 분석하여 실시간 컴퓨터 비전 파이프라인을 구축합니다.

### 🎯 학습 목표
1. **MediaPipe**와 **OpenCV**를 결합하여 실시간 실습에 필요한 프레임 처리 파이프라인(RGB 변환 및 렌더링) 구축
2. **Face Detection & Face Mesh**를 통해 얼굴의 바운딩 박스 및 468개 3D 랜드마크 시각화
3. **Hand Tracking**으로 손의 21개 관절을 정확하게 검출하고 손가락 간 거리 계산 구현
4. **Pose Estimation**으로 전신 33개 관절을 추출하고, 삼각함수를 활용한 주요 관절의 각도 계산 공식 적용

In [23]:
# OpenCV: 웹캠 입력/색상 변환/화면 출력
import cv2
# MediaPipe: 이미지 래핑/모델 실행
import mediapipe as mp
# 모델 경로/Tasks API 클래스 준비
from pathlib import Path
import urllib.request

from mediapipe.tasks import python
from mediapipe.tasks.python import vision

import numpy as np
from PIL import ImageFont, ImageDraw, Image

from config import CONTENT_DIR

In [12]:
# 모델 저장 폴더
MODEL_DIR = Path("models")
MODEL_DIR.mkdir(exist_ok=True)

# 얼굴 박스 감지 모델
FACE_DETECTOR_MODEL = MODEL_DIR / "blaze_face_short_range.tflite"

# 얼굴 랜드마크 모델
FACE_LANDMARKER_MODEL = MODEL_DIR / "face_landmarker.task"

# 공식 모델 URL
FACE_DETECTOR_MODEL_URL = "https://storage.googleapis.com/mediapipe-models/face_detector/blaze_face_short_range/float16/latest/blaze_face_short_range.tflite"
FACE_LANDMARKER_MODEL_URL = "https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/latest/face_landmarker.task"

def download_model(model_path, model_url):
    # 기존 모델은 다운로드 생략
    if model_path.exists():
        return

    # 모델이 없을 때만 다운로드
    print(f"모델 다운로드 중: {model_path}")
    urllib.request.urlretrieve(model_url, model_path)


# 첫 실행 시 모델 자동 다운로드
download_model(FACE_DETECTOR_MODEL, FACE_DETECTOR_MODEL_URL)
download_model(FACE_LANDMARKER_MODEL, FACE_LANDMARKER_MODEL_URL)

# Tasks API 클래스 별칭
BaseOptions = python.BaseOptions
VisionRunningMode = vision.RunningMode
FaceDetector = vision.FaceDetector
FaceDetectorOptions = vision.FaceDetectorOptions
FaceLandmarker = vision.FaceLandmarker
FaceLandmarkerOptions = vision.FaceLandmarkerOptions
FaceLandmarksConnections = vision.FaceLandmarksConnections
mp_drawing = vision.drawing_utils

def create_face_detector(min_detection_confidence=0.5):
    # 얼굴 박스/keypoint 감지기 생성
    options = FaceDetectorOptions(
        base_options=...,
        running_mode=...,
        min_detection_confidence=...,
    )
    return ...  # 감지기 생성


def normalized_to_pixel_coordinates(normalized_x, normalized_y, image_width, image_height):
    # 정규화 좌표 -> 픽셀 좌표 변환
    if not (0 <= normalized_x <= 1 and 0 <= normalized_y <= 1):
        return None

    # 이미지 범위를 벗어나지 않도록 마지막 픽셀 인덱스로 제한
    x = min(int(normalized_x * image_width), image_width - 1)
    y = min(int(normalized_y * image_height), image_height - 1)
    return x, y


def draw_face_detections(image, detection_result):
    # 감지 결과 표시
    height, width, _ = image.shape

    # 얼굴을 감싸는 바운딩 박스
    for detection in detection_result.detections:
        bbox = detection.bounding_box
        start_point = (bbox.origin_x, bbox.origin_y)
        end_point = (bbox.origin_x + bbox.width, bbox.origin_y + bbox.height)
        cv2.rectangle(image, start_point, end_point, (0, 0, 255), 2)

        # 눈, 코, 입 등 얼굴의 대표 지점
        for keypoint in detection.keypoints:
            keypoint_px = normalized_to_pixel_coordinates(keypoint.x, keypoint.y, width, height)
            if keypoint_px:
                cv2.circle(image, keypoint_px, 3, (0, 255, 0), -1)


        # 얼굴 감지 신뢰도 표시
        if detection.categories:
            score = detection.categories[0].score
            cv2.putText(
                image,
                f"face {score:.2f}",
                (bbox.origin_x, max(20, bbox.origin_y - 10)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                (0, 0, 255),
                2,
                cv2.LINE_AA,
            )

## 1. 얼굴 감지 (Face Detection) & Face Mesh 실습

### 1.1 Face Detection
MediaPipe `mp.solutions.face_detection` 모델을 사용하여 얼굴을 감지하고 바운딩 박스를 시각화합니다.

In [ ]:
# 0번 카메라 열기
cap = cv2.VideoCapture(0)

# FPS 없으면 30 사용
fps = cap.get(cv2.CAP_PROP_FPS)
if fps <= 0:
    fps = 30
frame_index = 0

# 모델 리소스 자동 정리용 with 문
with create_face_detector(min_detection_confidence=0.5) as face_detector:
    while cap.isOpened():
        # 웹캠 프레임 읽기
        res, image = cap.read()
        if not res:
            print('웹캠에서 이미지를 읽지 못했습니다.')
            break

        # BGR -> RGB 변환
        rgb_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        # MediaPipe 입력 이미지 생성
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_image)

        # 프레임 번호 -> timestamp 변환
        # VIDEO 모드에서는 프레임마다 증가하는 timestamp가 필요
        timestamp_ms = int(frame_index * 1000 / fps)
        frame_index += 1

        # 현재 프레임 얼굴 감지
        # face_detector : 웹캠/동영상처럼 연속된 프레임을 처리할 때 쓰는 감지 함
        detection_result = face_detector.detect_for_video(mp_image, timestamp_ms)

        # 얼굴 박스/keypoint 표시
        # 감지된 얼굴 바운딩박스와 keypoing를 원본 이미지 위에 그림.
        draw_face_detections(image, detection_result)

        # 결과 화면 출력
        cv2.imshow('frame', image)

        # q 입력 시 종료
        if cv2.waitKey(1) == ord('q'):
            break

# 카메라/창 정리
cap.release()
cv2.destroyAllWindows()

### 1.2 Face Mesh
얼굴 표면을 더욱 정밀하게 복원하기 위해 `mp.solutions.face_mesh` 모델을 활용하여 468개(혹은 478개) 3D 랜드마크를 추출합니다.

In [14]:
# 랜드마크 점/선 스타일
drawing_spec = mp_drawing.DrawingSpec(thickness=1, circle_radius=1)
def create_face_landmarker(num_faces=1):
    # 얼굴 랜드마크 검출기 생성
    options = FaceLandmarkerOptions(
        base_options = BaseOptions(model_asset_path=str(FACE_LANDMARKER_MODEL)),
        running_mode = VisionRunningMode.VIDEO,
        num_faces=num_faces,                # 한 프레임에서 최대 몇 명의 얼굴을 찾을지.
        min_face_detection_confidence=0.5,  # 얼굴 검출 최소 신뢰도 
        min_face_presence_confidence=0.5,   # 얼굴이 부분적으로 가려졌을 때 결과 안정성에 영향을 줌.
        min_tracking_confidence=0.5         # 값이 높으면 안정적 추척만 유지, 낮으면 추적이 더 오래 유지.
    )
    return FaceLandmarker.create_from_options(options)



In [ ]:
# 얼굴 메시 실시간 표시 예제
cap = cv2.VideoCapture(0)
fps = cap.get(cv2.CAP_PROP_FPS)
if fps <= 0:
    fps = 30
frame_index = 0

with create_face_landmarker(num_faces=1) as face_landmarker:
    while cap.isOpened():
        res, image = cap.read()
        if not res:
            print('웹캠에서 이미지를 읽지 못했습니다.')
            break

        # BGR -> RGB 변환
        rgb_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_image)

        # timestamp 계산
        timestamp_ms = int(frame_index * 1000 / fps)
        frame_index += 1

        # 얼굴 랜드마크 검출
        result =  face_landmarker.detect_for_video(mp_image, timestamp_ms)

        # result.face_landmarks : 얼굴마다 랜드마크 리스트를 담고 있음
        if result.face_landmarks:
            for face_landmarks in result.face_landmarks:
                # 얼굴 전체 메시 연결선 그리기
                mp_drawing.draw_landmarks(
                    image=image, # 랜드마크를 그릴 이미지
                    landmark_list=face_landmarks, # 검출된 얼굴 랜드마크 좌표들
                    connections=FaceLandmarksConnections.FACE_LANDMARKS_TESSELATION,
                    landmark_drawing_spec=drawing_spec,     # 점 스타일
                    connection_drawing_spec= drawing_spec   # 선 스타일
                )

        cv2.imshow("frame", image)
        if cv2.waitKey(1) == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()

## 2. 핸드 트래킹 (Hand Tracking) 및 관절 거리 분석

### 2.1 Hand Joint Landmark 추출
손가락의 21개 관절(`mp.solutions.hands`) 위치를 실시간으로 탐지합니다.

In [24]:
# 모델 저장 폴더
MODEL_DIR = Path("models")
MODEL_DIR.mkdir(exist_ok=True)

# 손 위치/21개 랜드마크 검출 모델
HAND_LANDMARKER_MODEL = MODEL_DIR / "hand_landmarker.task"
HAND_LANDMARKER_MODEL_URL = "https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/latest/hand_landmarker.task"

def download_model(model_path, model_url):
    # 기존 모델은 다운로드 생략
    if model_path.exists():
        return

    # 모델이 없을 때만 다운로드
    print(f"모델 다운로드 중: {model_path}")
    urllib.request.urlretrieve(model_url, model_path)


# 첫 실행 시 손 랜드마크 모델 다운로드
download_model(HAND_LANDMARKER_MODEL, HAND_LANDMARKER_MODEL_URL)

# Tasks API 클래스 별칭
BaseOptions = python.BaseOptions
VisionRunningMode = vision.RunningMode
HandLandmarker = vision.HandLandmarker
HandLandmarkerOptions = vision.HandLandmarkerOptions
mp_hands = vision.HandLandmarksConnections
mp_drawing = vision.drawing_utils

def create_hand_landmarker(num_hands=2):
    # 손 랜드마크 검출기 생성
    options = HandLandmarkerOptions(
        base_options=BaseOptions(model_asset_path=str(HAND_LANDMARKER_MODEL)),
        running_mode=VisionRunningMode.VIDEO,
        num_hands=num_hands,
        min_hand_detection_confidence=0.5, # 손이 잘 안 잡히면 낮추고, 배경을 손으로 잘못 잡으면 높인다.
        min_hand_presence_confidence=0.5,
        min_tracking_confidence=0.5
    )
    return HandLandmarker.create_from_options(options)


def draw_hand_landmarks(image, hand_result):
    # 손마다 21개 랜드마크 표시
    for hand_landmarks in hand_result.hand_landmarks:
        mp_drawing.draw_landmarks(
            image,
            hand_landmarks,                 # 검출된 손 랜드마크 
            mp_hands.HAND_CONNECTIONS,      # 손가락 관절 연결 정보
            mp_drawing.DrawingSpec(color=(121, 22, 76), thickness=2, circle_radius=4),  # 점 스타일
            mp_drawing.DrawingSpec(color=(250, 44, 250), thickness=2)   # 선 스타일
        )


def draw_text(img, text, position, font_size, font_color):

    # opencv 이미지를 PIL이미지로 변환
    img_pil = Image.fromarray(img)

    # PIL Draw 객체 생성
    draw = ImageDraw.Draw(img_pil)

    # position 위치에 text를 그림. font_color는 RGB 순서.
    draw.text(position, text, fill=font_color)

    # 다시 OpenCV에서 사용할 수 있도록 최종 numpy array 로 이미지 형태 반환.
    return np.array(img_pil)

def get_hand_pose(hand_landmarks, handedness=None):
    # MediaPipe 버전에 맞는 랜드마크 리스트 선택
    landmarks = hand_landmarks.landmark if hasattr(hand_landmarks, "landmark") else hand_landmarks

    open_fingers = []

    # 엄지 랜드마크 선택
    thumb_tip = landmarks[4]
    thumb_ip = landmarks[3]
    thumb_mcp = landmarks[2]
    pinky_mcp = landmarks[17]

    # 엄지 펼침 판정
    if thumb_mcp.x < pinky_mcp.x:
        open_fingers.append(thumb_tip.x < thumb_ip.x)
    else:
        open_fingers.append(thumb_tip.x > thumb_ip.x)

    # 검지~새끼 펼침 판정
    # Mediapipe 이미지 좌표계에서는 위쪽으로 갈수록 y값이 작아짐
    # 손끝 TIP의 y값이 PIP 관절보다 작으면 손가락이 위로 펴져있다. 
    for tip_idx, pip_idx in [(8, 6), (12, 10), (16, 14), (20,18)]:
        tip = landmarks[tip_idx]
        pip = landmarks[pip_idx]
        open_fingers.append(tip.y < pip.y)

    thumb_open, index_open, middle_open, ring_open, pinky_open = open_fingers

    # 펴진 손가락 개수를 셈
    open_count = open_fingers.count(True)

    # 가위/바위/보 판정
    if open_count == 0:
        return "rock"
    elif open_count == 5:
        return "paper"
    elif open_count == 2 and index_open:
        return "picking"
    else:
        return "unknown"

In [27]:
# 포즈 판별용 웹캠 루프
cap = cv2.VideoCapture(0)
fps = cap.get(cv2.CAP_PROP_FPS)
if fps <= 0:
    fps = 30
frame_index = 0

# 가위/바위/보는 한 손 기준
with create_hand_landmarker(num_hands=1) as hand_landmarker:
    while cap.isOpened():
        success, image = cap.read()
        if not success:
            continue

        rgb_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_image)

        timestamp_ms = int(frame_index * 1000 / fps)
        frame_index += 1

        # 손 랜드마크/handedness 검출
        result = hand_landmarker.detect_for_video(mp_image, timestamp_ms)

        if result.hand_landmarks:
            for idx, hand_landmarks in enumerate(result.hand_landmarks):
                # 손 관절 화면 표시
                mp_drawing.draw_landmarks(
                    image,
                    hand_landmarks,                 # 검출된 손 랜드마크 
                    mp_hands.HAND_CONNECTIONS,      # 손가락 관절 연결 정보
                    mp_drawing.DrawingSpec(color=(121, 22, 76), thickness=2, circle_radius=4),  # 점 스타일
                    mp_drawing.DrawingSpec(color=(250, 44, 250), thickness=2)   # 선 스타일
                )

                # 왼손/오른손 분류 결과
                # 손바닥을 카메라로 향했을 때 오른손 -> 엄지가 화면 왼쪽 / 왼손 -> 엄지가 화면 오른쪽
                handedness = result.handedness[idx] if idx < len(result.handedness) else None

                # 손가락 상태 -> 가위/바위/보 변환
                pose = get_hand_pose(hand_landmarks, handedness)

                # 한글 결과 출력
                image = draw_text(image, pose, (10, 50), 30, (255,255,255))

        cv2.imshow('Hand Pose', image)

        if cv2.waitKey(5) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()

I0000 00:00:1779092460.077092  201757 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1779092460.117136  201768 gl_context.cc:385] GL version: 3.0 (OpenGL ES 3.0 Mesa 23.2.1-1ubuntu3.1~22.04.3), renderer: D3D12 (Intel(R) Iris(R) Xe Graphics)
W0000 00:00:1779092460.245074  201760 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1779092460.341519  201766 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
[ WARN:0@11103.675] global cap_v4l.cpp:1049 tryIoctl VIDEOIO(V4L2:/dev/video0): select() timeout.
[ WARN:0@11113.688] global cap_v4l.cpp:1049 tryIoctl VIDEOIO(V4L2:/dev/video0): select() timeout.


KeyboardInterrupt: 

### 2.2 유클리드 거리를 활용한 제스처 정의
엄지 손가락 끝(`THUMB_TIP`, Index 4)과 검지 손가락 끝(`INDEX_FINGER_TIP`, Index 8) 사이의 거리를 계산하여 특정 집기(Pinch) 액션이나 간단한 제스처를 정의합니다.

**거리 계산 공식 (Euclidean Distance)**:
$$d = \sqrt{(x_1 - x_2)^2 + (y_1 - y_2)^2 + (z_1 - z_2)^2}$$

In [ ]:
# [실습] 두 손가락 끝의 3차원 좌표 간의 유클리드 거리를 계산하여 화면에 표시하는 기능을 구현하세요.

## 3. 자세 감지 (Pose Estimation) 및 관절 각도 계산

### 3.1 Pose Landmarks 추출
전신 33개 랜드마크(`mp.solutions.pose`)를 추출하여 신체 구조적 구도를 분석합니다.

In [ ]:
mp_pose = mp.solutions.pose

# [실습] 입력 비디오 스트림에서 전신 자세 랜드마크를 추적하는 파이프라인을 구축하세요.

### 3.2 랜드마크 기반 관절 각도(Angle) 계산
어깨(Shoulder), 팔꿈치(Elbow), 손목(Wrist)의 세 랜드마크 좌표를 기반으로 벡터 내적을 이용하여 팔꿈치 각도를 계산합니다.

**각도 계산 공식 (삼각함수 활용)**:
$$v_1 = A - B, \quad v_2 = C - B$$
$$\theta = \arccos\left(\frac{v_1 \cdot v_2}{\|v_1\| \|v_2\|}\right) \times \frac{180}{\pi}$$

In [ ]:
def calculate_angle(a, b, c):
    """세 점 a, b, c 사이의 각도를 계산합니다 (b가 중심점)"""
    a = np.array(a) # First landmark
    b = np.array(b) # Mid landmark (Joint)
    c = np.array(c) # End landmark
    
    rad = np.arctan2(c[1] - b[1], c[0] - b[0]) - np.arctan2(a[1] - b[1], a[0] - b[0])
    angle = np.abs(rad * 180.0 / np.pi)
    
    if angle > 180.0:
        angle = 360.0 - angle
        
    return angle

# [실습] Pose Landmarks에서 어깨, 팔꿈치, 손목 좌표를 추출하고 calculate_angle 함수를 활용해 각도를 렌더링하세요.

# 실습01_안경씌우기
1. 얼굴을 탐지한다.
2. 안경을 씌우기 위해 눈의 위치를 탐색한다.
3. 안경을 씌우는 위치에 안경에 해당하는 이미지를 사이즈 조정하여 넣는다. 비트 연산하여 넣는다.
4. 안경이 씌워진 얼굴 이미지를 출력한다.
5. 추가 : 눈의 각도를 계산해서 안경 이미지도 회전하는 코드를 작성해본다.

In [2]:
# TODO: 안경 PNG 불러오기
glasses_img = cv2.imread(CONTENT_DIR / 'glasses.png', cv2.IMREAD_UNCHANGED)

# 파일 로드 확인
if glasses_img is None:
    raise FileNotFoundError('glasses.png 파일을 찾을 수 없습니다.')

# 알파 채널 보정
if glasses_img.shape[2] == 3:
    alpha = np.full(glasses_img.shape[:2] + (1,), 255, dtype=np.uint8)
    glasses_img = np.concatenate([glasses_img, alpha], axis=2)

print('안경 이미지 크기:', glasses_img.shape)
cv2.imshow('glass_img', glasses_img)
cv2.waitKey(0)
cv2.destroyAllWindows()

안경 이미지 크기: (353, 707, 4)


QFontDatabase: Cannot find font directory /home/hong/project/ai-camp-note/.venv/lib/python3.13/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/hong/project/ai-camp-note/.venv/lib/python3.13/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/hong/project/ai-camp-note/.venv/lib/python3.13/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/hong/project/ai-camp-note/.venv/lib/python3.13/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabas

In [14]:
# FaceLandmarker 모델 준비
# Tasks API로 얼굴 메시 실행
MODEL_DIR = Path('models')
MODEL_DIR.mkdir(exist_ok=True)

FACE_LANDMARKER_MODEL = MODEL_DIR / 'face_landmarker.task'
FACE_LANDMARKER_MODEL_URL = 'https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/latest/face_landmarker.task'

def download_model(model_path, model_url):
    # 기존 모델은 다운로드 생략
    if model_path.exists():
        return
    print(f'모델 다운로드 중: {model_path}')
    urllib.request.urlretrieve(model_url, model_path)


download_model(FACE_LANDMARKER_MODEL, FACE_LANDMARKER_MODEL_URL)

BaseOptions = python.BaseOptions
VisionRunningMode = vision.RunningMode
FaceLandmarker = vision.FaceLandmarker
FaceLandmarkerOptions = vision.FaceLandmarkerOptions

def create_face_landmarker(num_faces=1):
    # 1. 얼굴 랜드마크 감지에 필요한 상세 옵션 설정
    options = FaceLandmarkerOptions(
        # 다운로드받은 모델 파일 경로 전달 (.task 파일)
        base_options=BaseOptions(model_asset_path=str(FACE_LANDMARKER_MODEL)),
        
        # 웹캠/비디오 프레임용 모드로 동작하도록 설정 (VIDEO 모드)
        running_mode=VisionRunningMode.VIDEO,
        
        # 탐지할 최대 얼굴 수
        num_faces=num_faces,
        
        # 신뢰도 임계치 설정 (기본값 0.5 권장)
        min_face_detection_confidence=0.5,  # 얼굴 감지 최소 신뢰도
        min_face_presence_confidence=0.5,   # 얼굴 랜드마크 존재 최소 신뢰도
        min_tracking_confidence=0.5         # 프레임 간 추적 유지 최소 신뢰도
    )
    
    # 2. 완성된 옵션 규격으로 랜드마커 객체를 생성 및 반환
    return FaceLandmarker.create_from_options(options)


def to_mp_image(frame):
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    return mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)


def overlay_bgra(background, overlay, x, y):
    # BGRA 이미지 알파 블렌딩
    bg_h, bg_w = background.shape[:2]
    ov_h, ov_w = overlay.shape[:2]

    # 화면과 겹치는 영역 계산
    x1 = max(x, 0)
    y1 = max(y, 0)
    x2 = min(x + ov_w, bg_w)
    y2 = min(y + ov_h, bg_h)

    # TODO: 겹치는 영역 없으면 생략
    if x1 >= x2 or y1 >= y2:
        return background

    # overlay 겹침 영역 자르기
    ov_x1 = x1 - x
    ov_y1 = y1 - y
    ov_x2 = ov_x1 + (x2 - x1)
    ov_y2 = ov_y1 + (y2 - y1)

    overlay_crop = overlay[ov_y1:ov_y2, ov_x1:ov_x2]
    color = overlay_crop[:, :, :3].astype(np.float32)
    alpha = overlay_crop[:, :, 3:4].astype(np.float32) / 255.0

    roi = background[y1:y2, x1:x2].astype(np.float32)
    blended = alpha * color + (1.0 - alpha) * roi
    background[y1:y2, x1:x2] = blended.astype(np.uint8)
    return background

def rotate_bgra(image, angle_degrees):
    # 안경 이미지 중심 회전
    h, w = image.shape[:2]
    center = (w / 2, h / 2)
    matrix = cv2.getRotationMatrix2D(center, angle_degrees, 1.0)

    cos = abs(matrix[0, 0])
    sin = abs(matrix[0, 1])
    new_w = int((h * sin) + (w * cos))
    new_h = int((h * cos) + (w * sin))

    matrix[0, 2] += (new_w / 2) - center[0]
    matrix[1, 2] += (new_h / 2) - center[1]

    return cv2.warpAffine(
        image,
        matrix,
        (new_w, new_h),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=(0, 0, 0, 0),
    )

def eye_points(face_landmarks, frame_shape):
    # FaceLandmarker 좌표는 0~1 정규화
    frame_h, frame_w = frame_shape[:2]
    left_eye = face_landmarks[362]
    right_eye = face_landmarks[133]

    left = np.array([left_eye.x * frame_w, left_eye.y * frame_h])
    right = np.array([right_eye.x * frame_w, right_eye.y * frame_h])
    center = ((left + right) / 2).astype(int)
    distance = np.linalg.norm(left - right)
    angle = np.degrees(np.arctan2(left[1] - right[1], left[0] - right[0]))
    return left, right, center, distance, angle


def draw_glasses(frame, face_landmarks, rotate=False):
    # 눈 사이 거리 기준으로 안경 크기를 정합니다.
    _, _, center, eye_distance, angle = eye_points(face_landmarks, frame.shape)
    glasses_width = max(1, int(eye_distance * 3.7))

    # 회전을 고려하는 예제에서는 얼굴 기울기에 맞춰 안경 이미지를 먼저 회전합니다.
    source = rotate_bgra(glasses_img, -angle) if rotate else glasses_img
    scale = glasses_width / source.shape[1]
    glasses_height = max(1, int(source.shape[0] * scale))
    resized = cv2.resize(source, (glasses_width, glasses_height), interpolation=cv2.INTER_AREA)

    x = int(center[0] - glasses_width / 2)
    y = int(center[1] - glasses_height / 2)
    overlay_bgra(frame, resized, x, y)
    return frame

In [10]:
cap = cv2.VideoCapture(CONTENT_DIR / 'face_sample.mp4')

fps = cap.get(cv2.CAP_PROP_FPS)
if fps <= 0:
    fps = 30
frame_index = 0

with create_face_landmarker(num_faces=1) as face_landmarker:
    while cap.isOpened():
        success, frame = cap.read()
        if not success:
            print('웹캠에서 이미지를 읽지 못했습니다.')
            break

        mp_image = to_mp_image(frame)
        timestamp_ms = int(frame_index * 1000 / fps)
        frame_index += 1

        result = face_landmarker.detect_for_video(mp_image, timestamp_ms)

        if result.face_landmarks:
            for face_landmarks in result.face_landmarks:
                                # 1. 카메라 프레임(배경)의 세로, 가로 해상도 획득
                h, w, _ = frame.shape
                
                # 2. 왼쪽 눈(33번)과 오른쪽 눈(263번)의 상대 비율 좌표를 픽셀 좌표로 변환
                left_eye = face_landmarks[33]
                right_eye = face_landmarks[263]
                
                left_x, left_y = int(left_eye.x * w), int(left_eye.y * h)
                right_x, right_y = int(right_eye.x * w), int(right_eye.y * h)
                
                # 3. 양 눈 사이의 실시간 가로 픽셀 거리 계산
                eye_distance = right_x - left_x
                if eye_distance <= 0:
                    continue  # 예외 상황(얼굴이 반만 잡혔을 때 등) 대비 스킵
                
                # 4. 양안 거리에 비례하여 안경의 실시간 가로 크기(Scale) 조절
                glasses_width = int(eye_distance * 2.2)
                
                # 안경 고유 비율(Aspect Ratio)을 유지하여 세로 길이 계산
                aspect_ratio = glasses_img.shape[0] / glasses_img.shape[1]
                glasses_height = int(glasses_width * aspect_ratio)
                
                # 5. 실시간 비율 계산이 끝난 사이즈로 안경 PNG 리사이징
                resized_glasses = cv2.resize(glasses_img, (glasses_width, glasses_height))
                
                # 6. 안경의 정중앙을 안착시킬 미간(양 눈의 중심점) 픽셀 좌표 획득
                center_x = (left_x + right_x) // 2
                center_y = (left_y + right_y) // 2
                
                # 7. 안경 좌상단 모서리(x, y) 좌표 역산 (안경 중심을 미간 위치에 고정)
                overlay_x = center_x - (glasses_width // 2)
                overlay_y = center_y - (glasses_height // 2)
                
                # 8. 투명도(알파) 마스킹 처리가 완료된 overlay_bgra 함수를 호출하여 안경 합성
                frame = overlay_bgra(frame, resized_glasses, overlay_x, overlay_y)


        cv2.imshow('Glasses Overlay', frame)
        if cv2.waitKey(5) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()

W0000 00:00:1779081606.352211  132010 face_landmarker_graph.cc:174] Sets FaceBlendshapesGraph acceleration to xnnpack by default.
I0000 00:00:1779081606.358801  132010 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1779081606.381449  132023 gl_context.cc:385] GL version: 3.0 (OpenGL ES 3.0 Mesa 23.2.1-1ubuntu3.1~22.04.3), renderer: D3D12 (Intel(R) Iris(R) Xe Graphics)
W0000 00:00:1779081606.384127  132014 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1779081606.408145  132015 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


In [15]:
cap = cv2.VideoCapture(CONTENT_DIR / 'face_sample.mp4')
fps = cap.get(cv2.CAP_PROP_FPS)
if fps <= 0:
    fps = 30
frame_index = 0

with create_face_landmarker(num_faces=1) as face_landmarker:
    while cap.isOpened():
        success, frame = cap.read()
        if not success:
            print('웹캠에서 이미지를 읽지 못했습니다.')
            break

        mp_image = to_mp_image(frame)
        timestamp_ms = int(frame_index * 1000 / fps)
        frame_index += 1

        result = face_landmarker.detect_for_video(mp_image, timestamp_ms)

        if result.face_landmarks:
            for face_landmarks in result.face_landmarks:
                # TODO: 눈 거리/중심으로 안경 합성
                draw_glasses(frame, face_landmarks, rotate=False)

        cv2.imshow('Glasses Overlay', frame)
        if cv2.waitKey(5) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()

W0000 00:00:1779081806.506987  133372 face_landmarker_graph.cc:174] Sets FaceBlendshapesGraph acceleration to xnnpack by default.
I0000 00:00:1779081806.511092  133372 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1779081806.534453  133383 gl_context.cc:385] GL version: 3.0 (OpenGL ES 3.0 Mesa 23.2.1-1ubuntu3.1~22.04.3), renderer: D3D12 (Intel(R) Iris(R) Xe Graphics)
W0000 00:00:1779081806.537403  133374 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1779081806.558749  133375 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
